<a href="https://colab.research.google.com/github/Simon666Z/financial-engineering-club-fall26/blob/KongResearch/Yanzhi_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
print("hello world")

hello world


## **Download Dataset and Processing**

In [3]:
!pip install -q yfinance pyarrow lxml
import requests
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Pretend the request is coming from a normal browser
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

# Extract the S&P 500 table from the HTML
sp500 = pd.read_html(StringIO(response.text))[0]

# Keep the useful columns
sp500 = sp500[
    [
        "Symbol",
        "Security",
        "GICS Sector",
        "GICS Sub-Industry"
    ]
].copy()

# Yahoo Finance formatting:
# BRK.B -> BRK-B
# BF.B  -> BF-B
sp500["Ticker"] = (
    sp500["Symbol"]
    .str.replace(".", "-", regex=False)
)

tickers = sp500["Ticker"].tolist()

print("Number of securities:", len(tickers))
display(sp500.head())

data = yf.download(
    tickers=tickers,
    start="2016-01-01",
    end="2026-01-01",
    interval="1d",
    auto_adjust=False,
    group_by="ticker",
    threads=True,
    progress=True
)

valid_tickers = []
invalid_tickers = []

available_tickers = set(
    data.columns.get_level_values(0)
)

for ticker in tickers:

    if ticker not in available_tickers:
        invalid_tickers.append(ticker)
        continue

    try:
        prices = data[(ticker, "Adj Close")].dropna()

        if len(prices) > 0:
            valid_tickers.append(ticker)
        else:
            invalid_tickers.append(ticker)

    except KeyError:
        invalid_tickers.append(ticker)


print("Valid tickers:", len(valid_tickers))
print("Invalid tickers:", invalid_tickers)

data = data.loc[
    :,
    data.columns.get_level_values(0).isin(valid_tickers)
]

panel = (
    data
    .stack(level=0, future_stack=True)
    .rename_axis(["Date", "Ticker"])
    .reset_index()
)

panel.columns.name = None

panel["Date"] = pd.to_datetime(panel["Date"])

panel = (
    panel
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

display(panel.head())

print("Number of rows:", len(panel))
print("Number of stocks:", panel["Ticker"].nunique())
print("Start:", panel["Date"].min())
print("End:", panel["Date"].max())

print("\nColumns:")
print(panel.columns.tolist())

metadata_cols = [
    c for c in [
        "Ticker",
        "Security",
        "GICS Sector",
        "GICS Sub-Industry"
    ]
    if c in sp500.columns
]

metadata = sp500[metadata_cols].drop_duplicates("Ticker")

panel = panel.merge(
    metadata,
    on="Ticker",
    how="left"
)

panel = (
    panel
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

g = panel.groupby("Ticker")

panel["ret_1d"] = (
    g["Adj Close"]
    .pct_change(fill_method=None)
)

panel["mom_5d"] = (
    g["Adj Close"]
    .pct_change(5, fill_method=None)
)

panel["mom_20d"] = (
    g["Adj Close"]
    .pct_change(20, fill_method=None)
)

panel["mom_60d"] = (
    g["Adj Close"]
    .pct_change(60, fill_method=None)
)

panel["vol_5d"] = (
    panel
    .groupby("Ticker")["ret_1d"]
    .transform(
        lambda x: x.rolling(5).std()
    )
)

panel["vol_20d"] = (
    panel
    .groupby("Ticker")["ret_1d"]
    .transform(
        lambda x: x.rolling(20).std()
    )
)

panel["volume_ma20"] = (
    panel
    .groupby("Ticker")["Volume"]
    .transform(
        lambda x: x.rolling(20).mean()
    )
)


# Cell 16
panel["volume_ratio_20d"] = (
    panel["Volume"]
    / panel["volume_ma20"]
)

panel["range_pct"] = (
    (panel["High"] - panel["Low"])
    / panel["Close"]
)

panel["intraday_ret"] = (
    panel["Close"]
    / panel["Open"]
    - 1
)

panel["target"] = (
    panel
    .groupby("Ticker")["ret_1d"]
    .shift(-1)
)

features = [
    "ret_1d",
    "mom_5d",
    "mom_20d",
    "mom_60d",
    "vol_5d",
    "vol_20d",
    "volume_ratio_20d",
    "range_pct",
    "intraday_ret"
]

Number of securities: 503


,Symbol,Security,GICS Sector,GICS Sub-Industry,Ticker
0,MMM,3M,Industrials,Industrial Conglomerates,MMM
1,AOS,A. O. Smith,Industrials,Building Products,AOS
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,ABT
3,ABBV,AbbVie,Health Care,Biotechnology,ABBV
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,ACN


[*********************100%***********************]  503 of 503 completed
ERROR:yfinance:
3 Failed downloads:
ERROR:yfinance:['VYLR', 'FDXF', 'HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2016-01-01 -> 2026-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1451624400, endDate = 1767243600")')


Valid tickers: 500
Invalid tickers: ['FDXF', 'HONA', 'VYLR']


,Date,Ticker,Open,High,Low,Close,Adj Close,Volume
0,2016-01-04,A,41.060001,41.189999,40.340000,40.689999,37.356037,3287300.0
1,2016-01-05,A,40.730000,40.950001,40.340000,40.549999,37.227524,2587200.0
2,2016-01-06,A,40.240002,40.990002,40.049999,40.730000,37.392761,2103600.0
3,2016-01-07,A,40.139999,40.150002,38.810001,39.000000,35.804501,3504300.0
4,2016-01-08,A,39.220001,39.709999,38.470001,38.590000,35.428101,3736700.0


Number of rows: 1257000
Number of stocks: 500
Start: 2016-01-04 00:00:00
End: 2025-12-31 00:00:00

Columns:
['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']


## **Define Additional Features**

In [4]:
work = (
    panel
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
    .copy()
)

g = work.groupby("Ticker", sort=False)

# 1–2. 10日和120日动量
work["mom_10d"] = g["Adj Close"].pct_change(
    10, fill_method=None
)
work["mom_120d"] = g["Adj Close"].pct_change(
    120, fill_method=None
)

# 3. 两日反转：近期下跌时，该值为正
work["reversal_2d"] = -g["Adj Close"].pct_change(
    2, fill_method=None
)

# 4. 过去60日的日回报率波动
work["vol_60d"] = g["ret_1d"].transform(
    lambda x: x.rolling(60).std()
)

# 5. 当前价格相对于过去60日最高价的回撤
rolling_high_60d = g["Adj Close"].transform(
    lambda x: x.rolling(60).max()
)
work["drawdown_60d"] = (
    work["Adj Close"] / rolling_high_60d - 1
)

# 6. 当日成交量比过去20日平均值高出多少个标准差
volume_std_20d = g["Volume"].transform(
    lambda x: x.rolling(20).std()
)
work["volume_z20"] = (
    (work["Volume"] - work["volume_ma20"])
    / volume_std_20d.replace(0, np.nan)
)

# 7. 收盘价在当日最高价和最低价之间的位置
daily_range = (work["High"] - work["Low"]).replace(0, np.nan)
work["close_position"] = (
    (work["Close"] - work["Low"]) / daily_range
)

# 8. 过去20日的下行波动
work["downside_vol_20d"] = g["ret_1d"].transform(
    lambda x: (
        x.clip(upper=0)
         .pow(2)
         .rolling(20)
         .mean()
         .pow(0.5)
    )
)

new_features = [
    "mom_10d",
    "mom_120d",
    "reversal_2d",
    "vol_60d",
    "drawdown_60d",
    "volume_z20",
    "close_position",
    "downside_vol_20d",
]

features_rf = features + new_features

# target 对应的是哪一个交易日；用于检查训练/测试边界
work["target_date"] = g["Date"].shift(-1)

work = work.replace([np.inf, -np.inf], np.nan)

model_rows = work.dropna(
    subset=features_rf + ["target", "target_date"]
).copy()

print("Total features:", len(features_rf))
print("Usable labeled rows:", len(model_rows))
display(work[["Date", "Ticker"] + new_features + ["target"]].tail())

Total features: 17
Usable labeled rows: 1154469


,Date,Ticker,mom_10d,mom_120d,reversal_2d,vol_60d,drawdown_60d,volume_z20,close_position,downside_vol_20d,target
1256995,2025-12-24,ZTS,0.062844,-0.195492,-0.013815,0.022566,-0.146432,-1.009071,0.923953,0.009547,0.005897
1256996,2025-12-26,ZTS,0.072837,-0.194594,-0.021774,0.022574,-0.141398,-0.781447,0.940792,0.009540,-0.001980
1256997,2025-12-29,ZTS,0.061689,-0.198576,-0.003905,0.022574,-0.143099,-0.495322,0.325586,0.009551,0.003413
1256998,2025-12-30,ZTS,0.044883,-0.198524,-0.001426,0.022585,-0.140174,-0.811561,0.446514,0.009412,-0.004667
1256999,2025-12-31,ZTS,0.021018,-0.182497,0.001270,0.022577,-0.144187,-0.820696,0.046732,0.008080,NaN


## **Train the Model**

### **Evaluate The Model**


In [6]:
import pandas as pd
from lightgbm import LGBMRegressor

cutoff = pd.Timestamp("2025-01-01")

train = model_rows[
    model_rows["target_date"] < cutoff
].copy()

test = model_rows[
    (model_rows["Date"] >= cutoff)
    & (model_rows["Date"] < pd.Timestamp("2026-01-01"))
].copy()

print("Training rows:", len(train))
print("Test rows:", len(test))
print("Training target ends:", train["target_date"].max())
print("Test starts:", test["Date"].min())

lgbm_params = {
    "objective": "regression",
    "n_estimators": 100,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": 10,
    "min_child_samples": 50,
    "subsample": 0.25,
    "subsample_freq": 1,
    "n_jobs": -1,
    "random_state": 42,
    "verbosity": -1,
}

lgbm = LGBMRegressor(**lgbm_params)

lgbm.fit(
    train[features_rf].astype("float32"),
    train["target"].astype("float32")
)

test["predicted_return"] = lgbm.predict(
    test[features_rf].astype("float32")
)

# 在每个交易日内部，按预测回报率从高到低排名
test["predicted_rank"] = (
    test.groupby("Date")["predicted_return"]
    .rank(ascending=False, method="first")
    .astype(int)
)

display(
    test[
        ["Date", "Ticker", "predicted_return",
         "predicted_rank", "target"]
    ]
    .sort_values(["Date", "predicted_rank"])
    .head(20)
)

Training rows: 1029902
Test rows: 124069
Training target ends: 2024-12-31 00:00:00
Test starts: 2025-01-02 00:00:00


,Date,Ticker,predicted_return,predicted_rank,target
1017920,2025-01-02,SMCI,0.033634,1,0.109152
143048,2025-01-02,BE,0.010128,2,0.040650
100310,2025-01-02,APP,0.008798,3,0.025952
781604,2025-01-02,MPWR,0.004882,4,0.043166
846968,2025-01-02,NUE,0.004821,5,0.008473
786632,2025-01-02,MRNA,0.004394,6,0.004286
965126,2025-01-02,RDDT,0.003790,7,0.071304
308972,2025-01-02,CVS,0.003698,8,0.035052
306458,2025-01-02,CVNA,0.003593,9,-0.112247
922388,2025-01-02,PLTR,0.003254,10,0.062508


## **Generate Ranking**

In [8]:
from lightgbm import LGBMRegressor

final_lgbm = LGBMRegressor(**lgbm_params)

final_lgbm.fit(
    model_rows[features_rf].astype("float32"),
    model_rows["target"].astype("float32")
)

last_date = work["Date"].max()

latest = (
    work[work["Date"] == last_date]
    .dropna(subset=features_rf)
    .copy()
)

latest["predicted_return"] = final_lgbm.predict(
    latest[features_rf].astype("float32")
)

ranking = (
    latest
    .sort_values("predicted_return", ascending=False)
    .reset_index(drop=True)
)

ranking.insert(0, "Rank", np.arange(1, len(ranking) + 1))

print("Ranking based on data through:", last_date.date())

display(
    ranking[
        ["Rank", "Ticker", "Security",
         "GICS Sector", "predicted_return"]
    ].head(30)
)

Ranking based on data through: 2025-12-31


,Rank,Ticker,Security,GICS Sector,predicted_return
0,1,LITE,Lumentum,Information Technology,0.004656
1,2,BE,Bloom Energy,Industrials,0.004589
2,3,SMCI,Supermicro,Information Technology,0.004023
3,4,HOOD,Robinhood Markets,Financials,0.003516
4,5,COIN,Coinbase,Financials,0.003364
5,6,MRNA,Moderna,Health Care,0.003341
6,7,P,Everpure,Information Technology,0.003328
7,8,SNDK,Sandisk,Information Technology,0.003077
8,9,SKYD,Skydance Corporation,Communication Services,0.002795
9,10,ARE,Alexandria Real Estate Equities,Real Estate,0.002463
